In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score

# 1. 数据加载和预处理
mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X = mnist.data.astype(np.float32) / 255.0  # 归一化到[0,1]
y = mnist.target.astype(np.int32)

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. 神经网络参数
input_size = 784
hidden_size = 256
output_size = 10
learning_rate = 0.001
epochs = 20
batch_size = 128

# 3. 参数初始化 (He初始化)
np.random.seed(42)
W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2. / input_size)
b1 = np.zeros(hidden_size)
W2 = np.random.randn(hidden_size, output_size) * np.sqrt(2. / hidden_size)
b2 = np.zeros(output_size)

# 4. 激活函数
def relu(x):
    return np.maximum(0, x)

def softmax(x):
    exps = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exps / np.sum(exps, axis=1, keepdims=True)

# 5. 训练记录
train_loss = []
train_acc = []
test_acc = []

# 6. 训练循环
for epoch in range(epochs):
    epoch_loss = 0
    correct = 0
    
    # 打乱数据顺序
    permutation = np.random.permutation(X_train.shape[0])
    X_train_shuffled = X_train[permutation]
    y_train_shuffled = y_train[permutation]
    
    for i in range(0, X_train.shape[0], batch_size):
        # 获取动态batch（修复关键错误点）
        X_batch = X_train_shuffled[i:i+batch_size]
        y_batch = y_train_shuffled[i:i+batch_size]
        current_batch_size = X_batch.shape[0]
        
        # 前向传播
        z1 = X_batch.dot(W1) + b1
        a1 = relu(z1)
        z2 = a1.dot(W2) + b2
        a2 = softmax(z2)
        
        # 计算损失（使用动态current_batch_size）
        loss = -np.mean(np.log(a2[np.arange(current_batch_size), y_batch] + 1e-8))
        epoch_loss += loss * current_batch_size
        
        # 反向传播
        grad_z2 = a2.copy()
        grad_z2[np.arange(current_batch_size), y_batch] -= 1
        grad_z2 /= current_batch_size  # 使用实际batch大小归一化
        
        grad_W2 = a1.T.dot(grad_z2)
        grad_b2 = np.sum(grad_z2, axis=0)
        
        grad_a1 = grad_z2.dot(W2.T)
        grad_z1 = grad_a1 * (z1 > 0)
        
        grad_W1 = X_batch.T.dot(grad_z1)
        grad_b1 = np.sum(grad_z1, axis=0)
        
        # 参数更新
        W1 -= learning_rate * grad_W1
        b1 -= learning_rate * grad_b1
        W2 -= learning_rate * grad_W2
        b2 -= learning_rate * grad_b2
        
        # 计算准确率
        pred = np.argmax(a2, axis=1)
        correct += np.sum(pred == y_batch)
    
    # 记录指标
    avg_loss = epoch_loss / X_train.shape[0]
    train_loss.append(avg_loss)
    train_acc.append(correct / X_train.shape[0])
    
    # 测试集评估
    z1_test = X_test.dot(W1) + b1
    a1_test = relu(z1_test)
    z2_test = a1_test.dot(W2) + b2
    test_pred = np.argmax(z2_test, axis=1)
    test_acc.append(accuracy_score(y_test, test_pred))
    
    print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Train Acc: {train_acc[-1]:.4f} | Test Acc: {test_acc[-1]:.4f}")

# 7. 可视化结果
plt.figure(figsize=(15,5))

plt.subplot(1,2,1)
plt.plot(train_loss)
plt.title("Training Loss")
plt.xlabel("Epoch")

plt.subplot(1,2,2)
plt.plot(train_acc, label='Train')
plt.plot(test_acc, label='Test')
plt.title("Accuracy")
plt.legend()
plt.show()

# 在混淆矩阵代码前添加以下内容

# 获取所有测试样本的预测结果
z1_test = X_test.dot(W1) + b1
a1_test = relu(z1_test)
z2_test = a1_test.dot(W2) + b2
test_pred = np.argmax(z2_test, axis=1)

# 随机选择9个样本（3正确6错误）
np.random.seed(42)
correct_idx = np.where(test_pred == y_test)[0]
wrong_idx = np.where(test_pred != y_test)[0]

show_correct = np.random.choice(correct_idx, 3, replace=False)
show_wrong = np.random.choice(wrong_idx, 6, replace=False)
# 完全随机选择9个样本
selected_idx = np.random.choice(len(y_test), 9, replace=False)

# 可视化结果
plt.figure(figsize=(12, 12))
for i, idx in enumerate(selected_idx):
    plt.subplot(3, 3, i+1)
    img = X_test[idx].reshape(28, 28) * 255  # 恢复像素值到0-255
    plt.imshow(img, cmap='gray')
    true_label = y_test[idx]
    pred_label = test_pred[idx]
    
    # 用颜色标注正确/错误
    color = 'green' if true_label == pred_label else 'red'
    plt.title(f"True: {true_label}\nPred: {pred_label}", color=color)
    plt.axis('off')

plt.suptitle(f"Model Accuracy: {accuracy_score(y_test, test_pred):.2%}", fontsize=14)
plt.tight_layout()
plt.show()

# 混淆矩阵
cm = confusion_matrix(y_test, test_pred)
plt.figure(figsize=(10,8))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()